In [1]:
import h5py
import pandas as pd
import numpy as np
import gc

def parse_archs4_table(archs4_file, root_key, table_key):
    # see https://maayanlab.cloud/archs4/help.html for more infos on how to navigate the file
    archs4 = h5py.File(archs4_file, 'r')
    
    data = dict()
    table = archs4[root_key][table_key]
    for key in table.keys():
        column_data_piece = table[key][0]
        if isinstance(column_data_piece, np.number):
            column_data = table[key][:]
        
        else:
            column_data = table[key].asstr()[:]
        
        data[key] = column_data
    
    return data


def filter_table_data(data, retain_keys):
    retain_data = {
        key: data[key].copy() for key in retain_keys
    }
    return retain_data


def get_srx_samn_from_items(relation_items):
    srx, samn = None, None
    for item in relation_items:
        if not item:
            continue
            
        key, value = item.split(': ')
            
        if key == 'SRA':
            srx = value.split('=')[-1]
        
        if key == 'BioSample':
            samn = value.split('/')[-1]
        
    return srx, samn


def extract_srx_and_samn_accessions(relation_data):
    srx_samn_accessions = {
        key: [] for key in ['srx_accessions', 'biosample_accessions']
    }
    for relation in relation_data:
        relation_items = relation.split(',')
        srx, samn = get_srx_samn_from_items(
            relation_items
        )
        srx_samn_accessions['srx_accessions'].append(srx)
        srx_samn_accessions['biosample_accessions'].append(samn)
    
    return srx_samn_accessions
        

data = parse_archs4_table(
    'human_gene_v2.2.h5',
    'meta',
    'samples'
)

retain_keys = [
    'geo_accession', 'molecule_ch1', 'organism_ch1', 'readsaligned', 'relation', 
    'sample', 'series_id', 'singlecellprobability', 'source_name_ch1', 'title'
]

filtered_data = filter_table_data(
    data, 
    retain_keys
)

srx_samn_accessions = extract_srx_and_samn_accessions(
    filtered_data['relation']
)

srx_samn_table = pd.DataFrame().from_dict(
    srx_samn_accessions,
    orient = 'columns'
)

table = pd.DataFrame().from_dict(
    filtered_data, 
    orient = 'columns'
)

table = pd.concat(
    [table, srx_samn_table],
    axis = 1
)

del data, filtered_data, srx_samn_accessions
gc.collect()

table

,geo_accession,molecule_ch1,organism_ch1,readsaligned,relation,sample,series_id,singlecellprobability,source_name_ch1,title,srx_accessions,biosample_accessions
0,GSM1000981,total RNA,Homo sapiens,102963402,SRA: https://www.ncbi.nlm.nih.gov/sra?term=SRX...,GSM1000981,GSE29282,0.007336,Human DLBCL cel line,OCI-LY1_48hrs_mRNAseq_3x_siNT_R1,SRX185895,SAMN01163911
1,GSM1000982,total RNA,Homo sapiens,85980162,SRA: https://www.ncbi.nlm.nih.gov/sra?term=SRX...,GSM1000982,GSE29282,-0.006492,Human DLBCL cel line,OCI-LY1_48hrs_mRNAseq_3x_siNT_R2,SRX185896,SAMN01163912
2,GSM1000983,total RNA,Homo sapiens,109810687,SRA: https://www.ncbi.nlm.nih.gov/sra?term=SRX...,GSM1000983,GSE29282,-0.006492,Human DLBCL cel line,OCI-LY1_48hrs_mRNAseq_3x_siNT_R3,SRX185897,SAMN01163913
3,GSM1000984,total RNA,Homo sapiens,105304785,SRA: https://www.ncbi.nlm.nih.gov/sra?term=SRX...,GSM1000984,GSE29282,0.007336,Human DLBCL cel line,OCI-LY1_48hrs_mRNAseq_3x_siBCL6_R1,SRX185898,SAMN01163914
4,GSM1000985,total RNA,Homo sapiens,97274296,SRA: https://www.ncbi.nlm.nih.gov/sra?term=SRX...,GSM1000985,GSE29282,0.030632,Human DLBCL cel line,OCI-LY1_48hrs_mRNAseq_3x_siBCL6_R2,SRX185899,SAMN01163915
...,...,...,...,...,...,...,...,...,...,...,...,...
722420,GSM999587,polyA RNA,Homo sapiens,1225510,SRA: https://www.ncbi.nlm.nih.gov/sra?term=SRX...,GSM999587,GSE40710,0.238154,Brain cortex,Parkinson's Disease PD 13,SRX185248,SAMN01163627
722421,GSM999588,polyA RNA,Homo sapiens,2283349,SRA: https://www.ncbi.nlm.nih.gov/sra?term=SRX...,GSM999588,GSE40710,0.009471,Brain cortex,Parkinson's Disease PD 14,SRX185249,SAMN01163628
722422,GSM999589,polyA RNA,Homo sapiens,3253397,SRA: https://www.ncbi.nlm.nih.gov/sra?term=SRX...,GSM999589,GSE40710,0.083552,Brain cortex,Parkinson's Disease PD 15,SRX185250,SAMN01163629
722423,GSM999590,polyA RNA,Homo sapiens,511,SRA: https://www.ncbi.nlm.nih.gov/sra?term=SRX...,GSM999590,GSE40710,0.333697,Brain cortex,Parkinson's Disease PD 16,SRX185251,SAMN01163630


In [3]:
table.to_csv(
    'archs4_sample_metadata.tsv.gz',
    sep = '\t',
    compression = 'gzip',
    index = False
)